## TRISAR - IEEE DATA FUSION CONTEST
#### Download data from Capella Space

### Imports

In [ ]:
from download_utils import (
    find_geo_scenes_near_point,
    download_tifs_from_csv,
    build_patch_dataset,
    build_dataset_csvs,
)

from pathlib import Path
import shutil

#### Find GEO scenes & download them
#### IMPORTANT: FOR TRAINING WE SUGGEST DOWNLOADING COUPLE OF REGIONS

#### USE only 1 region for validation!

In [13]:
# Use a loop with preset locations with lat lon coordinates :)
# we used 5-6 locations, this is an example:

LOCATIONS = [
    {
        "name": "oroville",
        "target_lat": 39.526979021,
        "target_lon": -121.513563007,
    },
    {
        "name": "sydney",
        "target_lat": -33.873962956,
        "target_lon": 151.220896493,
    },
    {
        "name": "san_jose_validation",
        "target_lat": 37.313149020,
        "target_lon": -121.894385880,
    },
]


DELTA_LAT = 0.1
DELTA_LON = 0.1
LIMIT_ROWS = 2 # the more the better, we used 10-30

CONTEST_COLLECTION = "Capella_IEEE_DataContest_2026.csv"
FINAL_DATA_ROOT = Path("final_data")
CSV_DIR = Path("site_csvs")

FINAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

In [14]:
for loc in LOCATIONS:
    site_name = loc["name"]
    site_csv = CSV_DIR / f"{site_name}.csv"
    site_out_dir = FINAL_DATA_ROOT / f"downloaded_geo_tifs_{site_name}"

    print(f"Processing site {site_name}...")

    find_geo_scenes_near_point(
        csv_path=CONTEST_COLLECTION,
        output_csv=site_csv,
        target_lat=loc["target_lat"],
        target_lon=loc["target_lon"],
        delta_lat=DELTA_LAT,
        delta_lon=DELTA_LON,
        max_rows=None,
        limit_rows=LIMIT_ROWS,
    )

    download_tifs_from_csv(
        input_csv=site_csv,
        out_dir=site_out_dir,
        timeout=120,
    )

Processing site oroville...
Loading metadata...
Filtering GEO scenes...
Computing distances...
Building tif urls...
Saved 2 rows to site_csvs/oroville.csv
Loading scene table...
Found 2 rows to download.
[1/2] Downloading CAPELLA_C14_SP_GEO_HH_20240817222159_20240817222224...
[2/2] Downloading CAPELLA_C10_SP_GEO_HH_20240808235903_20240808235927...
Download finished.
Processing site sydney...
Loading metadata...
Filtering GEO scenes...
Computing distances...
Building tif urls...
Saved 2 rows to site_csvs/sydney.csv
Loading scene table...
Found 2 rows to download.
[1/2] Downloading CAPELLA_C13_SP_GEO_HH_20241122101645_20241122101653...
[2/2] Downloading CAPELLA_C13_SP_GEO_HH_20241116122315_20241116122324...
Download finished.
Processing site san_jose_validation...
Loading metadata...
Filtering GEO scenes...
Computing distances...
Building tif urls...
Saved 2 rows to site_csvs/san_jose_validation.csv
Loading scene table...
Found 2 rows to download.
[1/2] Downloading CAPELLA_C13_SP_GEO_HH_

#### Build patch dataset (we used 512 x 512 patches)

In [15]:
build_patch_dataset(
    input_root="final_data",
    output_root="patch_dataset",
    patch_size=512,
    require_full_patch=True,
    skip_empty_patches=True,
    min_images_per_patch=2,
    nodata_override=None,
)

Found 3 area folders
Processing area downloaded_geo_tifs_oroville...
Using reference tif CAPELLA_C10_SP_GEO_HH_20240808235903_20240808235927.tif
Found 2 tif files
Reference grid: 44 rows x 45 cols
Processed 100 reference patches, kept 78 groups, removed 22 groups
Processed 200 reference patches, kept 173 groups, removed 27 groups
Processed 300 reference patches, kept 273 groups, removed 27 groups
Processed 400 reference patches, kept 373 groups, removed 27 groups
Processed 500 reference patches, kept 473 groups, removed 27 groups
Processed 600 reference patches, kept 573 groups, removed 27 groups
Processed 700 reference patches, kept 671 groups, removed 29 groups
Processed 800 reference patches, kept 761 groups, removed 39 groups
Finished area downloaded_geo_tifs_oroville
Processed reference patches: 846
Kept bbox groups: 799
Removed bbox groups: 47
Processing area downloaded_geo_tifs_san_jose_validation...
Using reference tif CAPELLA_C13_SP_GEO_HH_20250221065943_20250221065952.tif
Fou

#### Build .csv files

In [20]:
# USE ABSOLUTE PATH
PATCH_ROOT = Path("/projects/uavnavi/grss-challenge/TRISAR/data/patch_dataset")
CSV_ROOT = Path("/projects/uavnavi/grss-challenge/TRISAR/data/dataset_csv")
VALIDATION_NAME = "downloaded_geo_tifs_san_jose_validation"

build_dataset_csvs(
    patch_root=PATCH_ROOT,
    out_dir=CSV_ROOT,
    validation_area_names=VALIDATION_NAME,
    negatives_per_positive=2,
    seed=42,
)

Building image manifest...
Total images: 5766
Total bbox groups: 2883
Train images: 2856
Validation images: 2910
Building pairs...
Train pairs: 4284
Validation pairs: 4365
Building triplets...
Train triplets: 5712
Validation triplets: 5820
Dataset csv files saved to /projects/uavnavi/grss-challenge/TRISAR/data/dataset_csv
